In [1]:
import sys
import random
from tensorflow.compat.v1.keras import backend as KK
import os
import math
import gc
import psutil

from tensorflow import keras
from keras import layers,Model

import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
import imageio
from sklearn.model_selection import train_test_split
import pydot

from keras.models import Sequential
from keras.layers import Flatten, Dense, Activation,Lambda
from IPython.display import clear_output
from tensorflow.keras.callbacks import EarlyStopping
from keras.utils import plot_model
import scipy.io
from sklearn.utils import shuffle
import joblib
from sklearn.metrics import mean_squared_error

In [2]:
proc = psutil.Process(os.getpid())
gc.collect()

0

In [3]:
## randomly select N_budget trajectories from N_total trajectories (can be done with other strategies) with seed_number
def random_sampling(N_budget,N_total,seed_number):
    np.random.seed(seed_number)
    select_idx = np.random.choice(N_total,N_budget,replace = False)
    return select_idx

In [4]:
def log_min_max_scale(X,select_idx):
    logX = np.log(X)
    X_train = logX[:,:,select_idx]
    X_max = np.nanmax(X_train)
    X_min = np.nanmin(X_train)
    logX_scaled = (logX-X_min)/(X_max-X_min)-0.5
    return logX_scaled

## Loading dataset and preprocessing

In [5]:
kf_ups_5 = scipy.io.loadmat('../../data/1000kf.mat')['kf_ups_5']
## scaling of input ####################################################
log_kf = np.reshape(np.log10(kf_ups_5[[0,2],:,:]),(-1,kf_ups_5.shape[-1]))
params_kf = (log_kf-np.min(log_kf))/(np.max(log_kf)-np.min(log_kf)) - 0.5

In [6]:
params_res = scipy.io.loadmat('../../data/params')['params'] - 0.5
params = np.concatenate((params_kf,params_res),axis = 0)
print("input parameter shape:", params.shape)

input parameter shape: (14, 1000)


In [7]:
T = scipy.io.loadmat('../../data/QoI3')['Tmax']
print("T shape:", T.shape)

T shape: (200, 1, 1000)


In [8]:
L = scipy.io.loadmat('../../data/QoI3')['L06']
L = L[:,0]
print("L shape:", L.shape)

L shape: (1000,)


In [9]:
select_idx = scipy.io.loadmat('../../data/training_idx2')['idx_miniMax']# idx_random; idx_sparse; idx_Maxmin; idx_miniMax
select_idx = select_idx[:,0]
print("select_idx shape:", select_idx.shape)

select_idx shape: (100,)


In [10]:
T_scaled = log_min_max_scale(X = T,select_idx = select_idx)
print("T_scaled shape:", T_scaled.shape)

T_scaled shape: (200, 1, 1000)


## Test

In [11]:
def set_test(N_mem,N_rec,N_budget,X,L,params):
    #####################################################
    N_par = params.shape[0]
    N_MC = params.shape[1]
    d_of_x = 1 # number of variables
    d_input = N_mem *d_of_x + N_par # number of input nodes
    d_output = d_of_x # number of output nodes
    d_outputRNN = N_rec * d_output # total number of output nodes (including recurrence)

    n_data = N_mem + N_rec
    n_burst = 20 # number of segments select from each trajectory
    choice_of_subsampling = 1 # default setting 0: no subsampling; 1: subsampling; 2: select the first (n_data+n_burst) data

    n_hidden = 3 # number of hidden layers
    n_nodes = 10 # number of nodes per hidden layer
    n_epochs = 10_000 # number of epochs for training
    learning_rate = 1e-4 # learning rate
    N_seeds = 1 # number of seeds for ensemble learning
    batch_size = 256 # batch size
    #####################################################
    Model_upperdir = '06tanh_logTmax' +'_{}mem'.format(N_mem) +'_{}rec'.format(N_rec)  +'_miniMax' + \
                        '_{}budget'.format(N_budget)+'_{}'.format(choice_of_subsampling)
    ## Test random trajectory
    N_test = X.shape[2]
    par_test = params.T ##(1000,10)
    #####################################################
    # ENSEMBLE PREDICTION
    X_pred = np.copy(X)
    for m in range(1,10):
        display(m)
        pred = np.zeros((1,d_of_x*L[m]))
        pred[:,:(d_of_x*N_mem)] = np.reshape(X[:N_mem,:,m],(1,-1))
        for j in range(N_seeds):
            savedir = Model_upperdir+'/seed_{}'.format(j)
            model_name_seed = 'model_seed_{}'.format(j)
            vars()[model_name_seed] = tf.keras.models.load_model(savedir)
        for i in range(L[m]-N_mem):
            temp_step = np.zeros(shape=(1,d_of_x*N_rec,N_seeds))
            for l in range(N_seeds):
                model_name_seed = 'model_seed_{}'.format(l)
                temp_step[:,:,l] = vars()[model_name_seed].predict(np.concatenate((pred[:,i*d_of_x:((N_mem-1)*d_of_x+(i+1)*d_of_x)],par_test[m:m+1,:]),axis = 1),verbose = 0)
            pred[:,((N_mem-1)*d_of_x+(i+1)*d_of_x):((N_mem-1)*d_of_x+(i+2)*d_of_x)] = np.mean(temp_step[:,:d_of_x,:],axis=2)
        pred = np.reshape(pred,(1, L[m], d_of_x))
        pred = np.transpose(pred,[1,2,0])
        X_pred[:L[m],:,m] = pred[:,:,0]
        scipy.io.savemat(Model_upperdir+'/pred_{}.mat'.format(m), {"T_pred": X_pred[:,:,m]})
        del pred
        del temp_step
        gc.collect()
    return

In [12]:
_ = set_test(N_mem = 20,N_rec = 10,N_budget= 100,X = T_scaled,L = L,params = params)

1

2

3

4

5

6

7

8

9